In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

project_path = "/content/drive/MyDrive/Energies_Christmas_2025"
data_path = os.path.join(project_path, "updated_timetable.txt")

# Load data (let pandas infer delimiter)
df = pd.read_csv(data_path, sep=None, engine="python")

df.head()
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79104 entries, 0 to 79103
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Time        79104 non-null  object 
 1   variables1  79104 non-null  float64
 2   variables2  79104 non-null  float64
 3   variables3  79104 non-null  float64
 4   variables4  79104 non-null  float64
 5   variables5  79104 non-null  float64
dtypes: float64(5), object(1)
memory usage: 3.6+ MB


In [ ]:
df.columns=['datetime','irradiation_forecast','temperature_forecast','irradiation','temperature','power']
tdi = pd.to_datetime(df['datetime'], format='%d-%m-%y %H')
df.set_index(tdi, inplace=True)

start_date = '2014-01-01'
end_date = '2019-01-31'
# Filter the data for the specified date range
data = df[start_date:end_date]

data = data.drop(columns=['datetime','irradiation_forecast','temperature_forecast'])
data['power'] = data['power']/1000

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.1,
    subplot_titles=(
        "Hourly Solar PV Power Output (2014–2019)",
        "Hourly Solar Irradiation (2014–2019)"
    )
)

# Top subplot: PV Power
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data["power"],
        mode="lines",
        line=dict(width=1),
        name="PV Power"
    ),
    row=1, col=1
)

# Bottom subplot: Irradiation
fig.add_trace(
    go.Scatter(
        x=data.index,
        y=data["irradiation"],
        mode="lines",
        line=dict(width=1),
        name="Irradiation"
    ),
    row=2, col=1
)

fig.update_layout(
    template="plotly_white",
    width=1400,
    height=900,
    showlegend=False,
    font=dict(size=14),
)

# Y-axes
fig.update_yaxes(
    title_text="PV Power (kW)",
    row=1, col=1
)

fig.update_yaxes(
    title_text="Irradiation (W/m²)",
    row=2, col=1
)

# 🔑 CRITICAL FIX
# Remove x-axis title & ticks from TOP subplot completely
fig.update_xaxes(
    title_text='Time',
    showticklabels=True,
    tickfont=dict(size=13),
    row=1, col=1
)

# Keep x-axis ONLY on bottom subplot
fig.update_xaxes(
    title_text="Time",
    showticklabels=True,
    tickfont=dict(size=13),
    row=2, col=1
)

fig.show()


In [ ]:
import numpy as np

stats = {
    "Power zero fraction (%)": 100 * np.mean(data["power"] == 0),
    "Mean daytime power (kW)": data.loc[data["irradiation"] > 0, "power"].mean(),
    "Max power (kW)": data["power"].max(),
    "Max irradiation (W/m²)": data["irradiation"].max(),
}

stats

{'Power zero fraction (%)': np.float64(48.60213606174834),
 'Mean daytime power (kW)': np.float64(0.6777609111525463),
 'Max power (kW)': 4.66751967340801,
 'Max irradiation (W/m²)': 1127.75315713483}

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Sample for clarity and performance
sampled = data.sample(5000, random_state=42)

variables = ["irradiation", "temperature"]

fig = make_subplots(
    rows=1,
    cols=len(variables),
    subplot_titles=[f"{v.capitalize()} vs PV Power" for v in variables]
)

for i, var in enumerate(variables, start=1):
    fig.add_trace(
        go.Scatter(
            x=sampled[var],
            y=sampled["power"],
            mode="markers",
            marker=dict(opacity=0.4),
            name=var
        ),
        row=1,
        col=i
    )

fig.update_layout(
    title="Relationship Between Meteorological Variables and PV Power",
    template="plotly_white",
    showlegend=False,
    height=450
)

fig.update_xaxes(title_text="Random 5000 samples")
fig.update_yaxes(title_text="PV Power (kW)")

fig.show()

In [ ]:
# Enforce physical consistency: no power without irradiation
data.loc[data["irradiation"] == 0, "power"] = 0.0

In [ ]:
import plotly.express as px

sampled = data.sample(5000, random_state=42)

fig = px.scatter(
    sampled,
    x="irradiation",
    y="power",
    opacity=0.4,
    labels={
        "irradiation": "Irradiation (W/m²)",
        "power": "PV Power (kW)"
    },
    title="Relationship between Solar Irradiation and PV Power Output"
)

fig.update_layout(
    template="plotly_white",
    width=900,                 # control horizontal size
    height=420,                # control vertical size
    title_font_size=18,
    font=dict(size=14),
    xaxis=dict(
        title_font_size=16,
        tickfont_size=13
    ),
    yaxis=dict(
        title_font_size=16,
        tickfont_size=13
    )
)

fig.show()

In [ ]:
# Drop temperature
data = data.drop(columns=["temperature"])

In [ ]:
df_feat = data.copy()

# Power lags
df_feat["power_t-1"] = df_feat["power"].shift(1)
df_feat["power_t-2"] = df_feat["power"].shift(2)

# Irradiation lags
df_feat["irradiation_t-1"] = df_feat["irradiation"].shift(1)
df_feat["irradiation_t-2"] = df_feat["irradiation"].shift(2)

# Target (current power)
df_feat["power_t"] = df_feat["power"]

In [ ]:
df_feat

,irradiation,power,power_t-1,power_t-2,irradiation_t-1,irradiation_t-2,power_t
datetime,,,,,,,
2014-01-01 00:00:00,0.496861,0.0,NaN,NaN,NaN,NaN,0.0
2014-01-01 01:00:00,0.379389,0.0,0.0,NaN,0.496861,NaN,0.0
2014-01-01 02:00:00,0.257452,0.0,0.0,0.0,0.379389,0.496861,0.0
2014-01-01 03:00:00,0.209292,0.0,0.0,0.0,0.257452,0.379389,0.0
2014-01-01 04:00:00,0.019541,0.0,0.0,0.0,0.209292,0.257452,0.0
...,...,...,...,...,...,...,...
2019-01-31 19:00:00,1.346272,0.0,0.0,0.0,1.037107,1.006410,0.0
2019-01-31 20:00:00,1.288054,0.0,0.0,0.0,1.346272,1.037107,0.0
2019-01-31 21:00:00,1.344754,0.0,0.0,0.0,1.288054,1.346272,0.0


In [ ]:
df_feat = df_feat.dropna()

In [ ]:
feature_cols = [
    "power_t-1",
    "power_t-2",
    "irradiation_t-1",
    "irradiation_t-2",
]

X = df_feat[feature_cols]
y = df_feat["power_t"]

In [ ]:
X.head(), y.head()

(                     power_t-1  power_t-2  irradiation_t-1  irradiation_t-2
 datetime                                                                   
 2014-01-01 02:00:00        0.0        0.0         0.379389         0.496861
 2014-01-01 03:00:00        0.0        0.0         0.257452         0.379389
 2014-01-01 04:00:00        0.0        0.0         0.209292         0.257452
 2014-01-01 05:00:00        0.0        0.0         0.019541         0.209292
 2014-01-01 06:00:00        0.0        0.0         0.013420         0.019541,
 datetime
 2014-01-01 02:00:00    0.0
 2014-01-01 03:00:00    0.0
 2014-01-01 04:00:00    0.0
 2014-01-01 05:00:00    0.0
 2014-01-01 06:00:00    0.0
 Name: power_t, dtype: float64)

In [ ]:
split_ratio = 0.8
split_idx = int(len(df_feat) * split_ratio)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

In [ ]:
from sklearn.preprocessing import StandardScaler

# Feature scaler
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

# Target scaler
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled  = scaler_y.transform(y_test.values.reshape(-1, 1))

In [ ]:
import numpy as np

# Persistence prediction (use power_t-1 directly)
y_pred_persistence = X_test["power_t-1"].values

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

mse_persist = mean_squared_error(y_test, y_pred_persistence)
rmse_persist = np.sqrt(mse_persist)
mae_persist = mean_absolute_error(y_test, y_pred_persistence)

rmse_persist, mae_persist

(np.float64(0.45210312866563135), 0.22225858975015506)

In [ ]:
from sklearn.linear_model import LinearRegression

arx = LinearRegression()
arx.fit(X_train_scaled, y_train_scaled.ravel())

LinearRegression()

In [ ]:
y_pred_arx_scaled = arx.predict(X_test_scaled)
y_pred_arx = scaler_y.inverse_transform(
    y_pred_arx_scaled.reshape(-1, 1)
).ravel()

In [ ]:
rmse_arx = np.sqrt(mean_squared_error(y_test, y_pred_arx))
mae_arx  = mean_absolute_error(y_test, y_pred_arx)

rmse_arx, mae_arx


(np.float64(0.3828416075781776), 0.21201737332197848)

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled, y_train_scaled.ravel())

Ridge(alpha=10)

In [ ]:
y_pred_ridge_scaled = ridge.predict(X_test_scaled)
y_pred_ridge = scaler_y.inverse_transform(
    y_pred_ridge_scaled.reshape(-1, 1)
).ravel()

In [ ]:
y_pred_ridge.shape

(8914,)

In [ ]:
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge  = mean_absolute_error(y_test, y_pred_ridge)

rmse_ridge, mae_ridge

(np.float64(0.3829112935462702), 0.21206469643814232)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
def split_sequences(input_par, output, n_steps):
    X, y = [], []
    for i in range(len(input_par)):
        end_ix = i + n_steps
        if end_ix > len(input_par):
            break
        X.append(input_par[i:end_ix, :])
        y.append(output[end_ix - 1])
    return np.array(X), np.array(y)

# supervised table
df_sup = data.copy()
df_sup["power_t-1"] = df_sup["power"].shift(1)
df_sup["irradiation_t-1"] = df_sup["irradiation"].shift(1)
df_sup = df_sup.dropna()

input_par = df_sup[["irradiation_t-1", "power_t-1"]].values
output = df_sup["power"].values

n_steps = 2
X_seq, y_seq = split_sequences(input_par, output, n_steps)

print(X_seq.shape)  # MUST be (N, 2, 2)

(44566, 2, 2)


In [ ]:
df_sup

,irradiation,power,power_t-1,irradiation_t-1
datetime,,,,
2014-01-01 01:00:00,0.379389,0.0,0.0,0.496861
2014-01-01 02:00:00,0.257452,0.0,0.0,0.379389
2014-01-01 03:00:00,0.209292,0.0,0.0,0.257452
2014-01-01 04:00:00,0.019541,0.0,0.0,0.209292
2014-01-01 05:00:00,0.013420,0.0,0.0,0.019541
...,...,...,...,...
2019-01-31 19:00:00,1.346272,0.0,0.0,1.037107
2019-01-31 20:00:00,1.288054,0.0,0.0,1.346272
2019-01-31 21:00:00,1.344754,0.0,0.0,1.288054


In [ ]:
split_idx = int(0.8 * len(X_seq))

X_train_seq = X_seq[:split_idx]
X_test_seq  = X_seq[split_idx:]

y_train_seq = y_seq[:split_idx]
y_test_seq  = y_seq[split_idx:]

In [ ]:
from sklearn.preprocessing import StandardScaler

n_features = X_train_seq.shape[2]

scaler_X_seq = StandardScaler()

X_train_flat = X_train_seq.reshape(-1, n_features)
X_test_flat  = X_test_seq.reshape(-1, n_features)

X_train_seq_scaled = scaler_X_seq.fit_transform(X_train_flat).reshape(X_train_seq.shape)
X_test_seq_scaled  = scaler_X_seq.transform(X_test_flat).reshape(X_test_seq.shape)

scaler_y_seq = StandardScaler()
y_train_seq_scaled = scaler_y_seq.fit_transform(y_train_seq.reshape(-1, 1))
y_test_seq_scaled  = scaler_y_seq.transform(y_test_seq.reshape(-1, 1))

print(X_train_seq_scaled.shape)  # MUST be (N, 2, 2)

(35652, 2, 2)


In [ ]:
import torch

Xtr = torch.tensor(X_train_seq_scaled, dtype=torch.float32)
ytr = torch.tensor(y_train_seq_scaled, dtype=torch.float32)
Xte = torch.tensor(X_test_seq_scaled, dtype=torch.float32)

print(Xtr.shape)  # torch.Size([N, 2, 2])

torch.Size([35652, 2, 2])


In [ ]:
import torch.nn as nn

class DLinear(nn.Module):
    def __init__(self, n_steps, n_features):
        super().__init__()
        self.linear = nn.Linear(n_steps, 1)

    def forward(self, x):
        x = x.transpose(1, 2)   # (batch, features, steps)
        out = self.linear(x)    # (batch, features, 1)
        return out.sum(dim=1)   # (batch, 1)

In [ ]:
model_dlinear = DLinear(
    n_steps=Xtr.shape[1],
    n_features=Xtr.shape[2]
)

optimizer = torch.optim.Adam(model_dlinear.parameters(), lr=1e-1)
loss_fn = nn.MSELoss()

epochs = 50

for _ in range(epochs):
    optimizer.zero_grad()
    pred = model_dlinear(Xtr)
    loss = loss_fn(pred, ytr)
    loss.backward()
    optimizer.step()

In [ ]:
# with torch.no_grad():
#     y_pred_dlinear_scaled = model_dlinear(Xte).numpy()

# y_pred_dlinear = scaler_y.inverse_transform(
#     y_pred_dlinear_scaled
# ).ravel()

In [ ]:
with torch.no_grad():
    y_pred_dlinear_scaled = model_dlinear(Xte).numpy()

y_pred_dlinear = scaler_y_seq.inverse_transform(
    y_pred_dlinear_scaled
).ravel()


rmse_dlinear = np.sqrt(mean_squared_error(y_test, y_pred_dlinear))
mae_dlinear  = mean_absolute_error(y_test, y_pred_dlinear)

rmse_dlinear, mae_dlinear


(np.float64(0.41330176022455695), 0.26197789426566553)

In [ ]:
import numpy as np

def split_sequences(input_par, output, n_steps):
    X, y = [], []
    for i in range(len(input_par)):
        end_ix = i + n_steps
        if end_ix > len(input_par):
            break
        seq_x = input_par[i:end_ix, :]
        seq_y = output[end_ix - 1]
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)


In [ ]:
values = data[["power", "irradiation"]].values

n_steps = 2

df_sup = data.copy()

df_sup["power_t-1"] = df_sup["power"].shift(1)
df_sup["irradiation_t-1"] = df_sup["irradiation"].shift(1)
df_sup = df_sup.dropna()

input_par = df_sup[["irradiation_t-1", "power_t-1"]].values
output = df_sup["power"].values

X_seq, y_seq = split_sequences(
    input_par=input_par,
    output=output,
    n_steps=n_steps
)



In [ ]:
split_idx = int(0.8 * len(X_seq))

X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()

X_train_flat = X_train.reshape(-1, n_features)
X_test_flat  = X_test.reshape(-1, n_features)

X_train_scaled = scaler_X.fit_transform(X_train_flat).reshape(X_train.shape)
X_test_scaled  = scaler_X.transform(X_test_flat).reshape(X_test.shape)

In [ ]:
scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_test_scaled  = scaler_y.transform(y_test.reshape(-1, 1))

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense

model_vanilla = Sequential()
model_vanilla.add(
    LSTM(
        5,
        activation="relu",
        input_shape=(n_steps, n_features)
    )
)
model_vanilla.add(Dense(1))

model_vanilla.compile(
    optimizer="adam",
    loss="mse"
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



In [ ]:
history = model_vanilla.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=20,
    batch_size=24,
    verbose=0
)


In [ ]:
y_pred_scaled = model_vanilla.predict(X_test_scaled)

y_pred = scaler_y.inverse_transform(y_pred_scaled).ravel()

279/279 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [ ]:
y_pred.shape

(8914,)

In [ ]:
split_idx = int(0.8 * len(df_sup))
test_datetime = df_sup.index[split_idx:]
det_df = pd.DataFrame({
    "datetime": test_datetime[:len(y_test)],  # safeguard
    "Actual": y_test,
    "Persistence": y_pred_persistence,
    "ARX": y_pred_arx,
    "Ridge": y_pred_ridge,
    "DLinear": y_pred_dlinear,
    "LSTM": y_pred
})

In [ ]:
det_df["Actual"].shape, det_df["ARX"].shape

((8914,), (8914,))

In [ ]:
# --- Manually selected representative days (edit freely) ---
selected_days = [
    "2018-02-15",  # Winter
    "2018-04-15",  # Spring
    "2018-07-15",  # Summer
    "2018-10-15",  # Autumn
]

selected_days = [pd.to_datetime(d) for d in selected_days]

In [ ]:
day_slices = []

for d in selected_days:
    day_df = det_df[
        (det_df["datetime"] >= d) &
        (det_df["datetime"] < d + pd.Timedelta(days=1))
    ]
    if day_df.empty:
        raise ValueError(f"No data available for {d.date()}")
    day_slices.append((d.strftime("%Y-%m-%d"), day_df))

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Visual styles per model ---
style_map = {
    "Actual":      dict(color="black",  width=3.0, dash="solid",   marker=None),
    "Persistence": dict(color="gray",   width=2.0, dash="dot",     marker=None),
    "ARX":         dict(color="green",  width=2.0, dash="dash",    marker="star"),
    "Ridge":       dict(color="blue",   width=2.0, dash="dashdot", marker=None),
    "DLinear":     dict(color="orange", width=2.0, dash="longdash",marker=None),
    "LSTM":        dict(color="red",    width=2.5, dash="solid",   marker=None),
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[day for day, _ in day_slices]
)

for i, (_, day_df) in enumerate(day_slices, start=1):
    r, c = (1,1) if i==1 else (1,2) if i==2 else (2,1) if i==3 else (2,2)

    for model, style in style_map.items():

        # 🔴 IMPORTANT: enable markers only when requested
        mode = "lines+markers" if style["marker"] else "lines"

        fig.add_trace(
            go.Scatter(
                x=day_df["datetime"],
                y=day_df[model],
                name=model,
                mode=mode,
                line=dict(
                    color=style["color"],
                    width=style["width"],
                    dash=style["dash"]
                ),
                marker=dict(
                    symbol=style["marker"],
                    size=10,
                    color=style["color"],
                    line=dict(color="black", width=1.5)
                ) if style["marker"] else None,
                showlegend=(i == 1),
                legendgroup=model
            ),
            row=r, col=c
        )

    fig.update_xaxes(
        tickformat="%H:%M",
        dtick=2 * 60 * 60 * 1000,
        title_text="Time (h)",
        row=r, col=c
    )

    fig.update_yaxes(
        title_text="PV Power (kW)",
        row=r, col=c
    )

fig.update_layout(
    height=1100,
    width=1900,
    template="simple_white",
    title=dict(
        text="Deterministic Hour-Ahead PV Power Forecasts on Representative Days",
        x=0.5,
        font=dict(size=24)
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(size=22)
    ),
    font=dict(size=22)
)

for ann in fig.layout.annotations:
    ann.font.size = 22

fig.show()

In [ ]:
def winkler_score(y, lower, upper, alpha=0.1):
    scores = []
    for yt, lt, ut in zip(y, lower, upper):
        width = ut - lt
        if yt < lt:
            score = width + (2 / alpha) * (lt - yt)
        elif yt > ut:
            score = width + (2 / alpha) * (yt - ut)
        else:
            score = width
        scores.append(score)
    return np.mean(scores)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred))
mae_lstm  = mean_absolute_error(y_test, y_pred)

rmse_lstm, mae_lstm


(np.float64(0.3335499489021375), 0.16059999594393504)

In [ ]:
def crps_interval(y, yhat, lower, upper):
    scores = []
    for yt, yht, lt, ut in zip(y, yhat, lower, upper):
        width = ut - lt
        if lt <= yt <= ut:
            score = width / 2
        else:
            score = abs(yt - yht) + width / 2
        scores.append(score)
    return np.mean(scores)

In [ ]:
def aci_full_test_daily_reset(
    y_true,
    y_pred,
    alpha=0.1,
    gamma=0.01,
    y_reset_value=0.0,
    warmup=50
):
    """
    Online ACI applied over the full test set.
    Intervals are produced sequentially and alpha is updated online.
    Daily reset is applied when y_true == y_reset_value.
    """

    T = len(y_true)
    lower = np.zeros(T)
    upper = np.zeros(T)

    alpha_t = alpha
    scores = []

    for t in range(T):

        # reset at nighttime / zero PV
        if y_true[t] == y_reset_value:
            alpha_t = alpha

        # warm-up: no update yet
        if t < warmup:
            q = np.quantile(
                np.abs(y_true[:t+1] - y_pred[:t+1]),
                1 - alpha
            )
        else:
            q = np.quantile(scores, 1 - alpha_t)

        lower[t] = y_pred[t] - q
        upper[t] = y_pred[t] + q

        # observe y_t and update alpha
        covered = (y_true[t] >= lower[t]) and (y_true[t] <= upper[t])
        scores.append(abs(y_true[t] - y_pred[t]))

        alpha_t = alpha_t + gamma * (alpha - (1 if not covered else 0))
        alpha_t = np.clip(alpha_t, 0.001, 0.999)

    return lower, upper

In [ ]:
def mpiw(lower, upper):
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    return np.mean(upper - lower)


def picp(y_true, lower, upper):
    y_true = np.asarray(y_true)
    lower = np.asarray(lower)
    upper = np.asarray(upper)

    return np.mean((y_true >= lower) & (y_true <= upper))


In [ ]:
lower_ridge, upper_ridge = aci_full_test_daily_reset(
    y_true=y_test,
    y_pred=y_pred_ridge,
    alpha=0.1,
    gamma=0.01
)

picp_ridge = picp(y_test, lower_ridge, upper_ridge)
mpiw_ridge = mpiw(lower_ridge, upper_ridge)
winkler_ridge = winkler_score(
    y_test, lower_ridge, upper_ridge, alpha=0.1
)
crps_ridge = crps_interval(
    y_test, y_pred_ridge, lower_ridge, upper_ridge
)

picp_ridge, mpiw_ridge, winkler_ridge, crps_ridge

(np.float64(0.9105900830154813),
 np.float64(1.1662969837929955),
 np.float64(1.995326684877563),
 np.float64(0.6773802111708228))

In [ ]:
lower_arx, upper_arx = aci_full_test_daily_reset(
    y_true=y_test,
    y_pred=y_pred_arx,
    alpha=0.1,
    gamma=0.01
)

picp_arx = picp(y_test, lower_arx, upper_arx)
mpiw_arx = mpiw(lower_arx, upper_arx)
winkler_arx = winkler_score(
    y_test, lower_arx, upper_arx, alpha=0.1
)
crps_arx = crps_interval(
    y_test, y_pred_arx, lower_arx, upper_arx
)

picp_arx, mpiw_arx, winkler_arx, crps_arx

(np.float64(0.9105900830154813),
 np.float64(1.1651065776516134),
 np.float64(1.9946636481059645),
 np.float64(0.6767943681604572))

In [ ]:
lower_dlinear, upper_dlinear = aci_full_test_daily_reset(
    y_true=y_test,
    y_pred=y_pred_dlinear,
    alpha=0.1,
    gamma=0.01
)

picp_dlinear = picp(y_test, lower_dlinear, upper_dlinear)
mpiw_dlinear = mpiw(lower_dlinear, upper_dlinear)
winkler_dlinear = winkler_score(
    y_test, lower_dlinear, upper_dlinear, alpha=0.1
)
crps_dlinear = crps_interval(
    y_test, y_pred_dlinear, lower_dlinear, upper_dlinear
)

picp_dlinear, mpiw_dlinear, winkler_dlinear, crps_dlinear

(np.float64(0.9158626879066637),
 np.float64(1.3794527009058974),
 np.float64(2.1255688812699622),
 np.float64(0.784855142540014))

In [ ]:
lower_lstm, upper_lstm = aci_full_test_daily_reset(
    y_true=y_test,
    y_pred=y_pred,        # LSTM predictions
    alpha=0.1,
    gamma=0.01
)

picp_lstm = picp(y_test, lower_lstm, upper_lstm)
mpiw_lstm = mpiw(lower_lstm, upper_lstm)
winkler_lstm = winkler_score(
    y_test, lower_lstm, upper_lstm, alpha=0.1
)
crps_lstm = crps_interval(
    y_test, y_pred, lower_lstm, upper_lstm
)

picp_lstm, mpiw_lstm, winkler_lstm, crps_lstm

(np.float64(0.9149652232443347),
 np.float64(1.0643628181152438),
 np.float64(1.7377698781978026),
 np.float64(0.6116721160465256))

In [ ]:
lower_pers, upper_pers = aci_full_test_daily_reset(
    y_true=y_test,
    y_pred=y_pred_persistence,
    alpha=0.1,
    gamma=0.01
)

picp_pers = picp(y_test, lower_pers, upper_pers)
mpiw_pers = mpiw(lower_pers, upper_pers)
winkler_pers = winkler_score(
    y_test, lower_pers, upper_pers, alpha=0.1
)
crps_pers = crps_interval(
    y_test, y_pred_persistence, lower_pers, upper_pers
)

picp_pers, mpiw_pers, winkler_pers, crps_pers

(np.float64(0.909356069104779),
 np.float64(1.6344979298666464),
 np.float64(2.3526365114337633),
 np.float64(0.9273240185578842))

In [ ]:
# --- Manually selected representative days (edit freely) ---
selected_days = [
    "2018-02-10",  # Winter
    "2018-06-08",  # Summer
]

selected_days = [pd.to_datetime(d) for d in selected_days]

In [ ]:
prob_df = pd.DataFrame({
    "datetime": test_datetime[:len(y_test)],

    "Actual": y_test,

    # ARX
    "ARX_pred": y_pred_arx,
    "ARX_lower": lower_arx,
    "ARX_upper": upper_arx,

    # Ridge
    "Ridge_pred": y_pred_ridge,
    "Ridge_lower": lower_ridge,
    "Ridge_upper": upper_ridge,

    # DLinear
    "DLinear_pred": y_pred_dlinear,
    "DLinear_lower": lower_dlinear,
    "DLinear_upper": upper_dlinear,

    # LSTM
    "LSTM_pred": y_pred,
    "LSTM_lower": lower_lstm,
    "LSTM_upper": upper_lstm,
})

In [ ]:
day_slices = []

for d in selected_days:
    day_df = prob_df[
        (prob_df["datetime"] >= d) &
        (prob_df["datetime"] < d + pd.Timedelta(days=1))
    ]
    if day_df.empty:
        raise ValueError(f"No data available for {d.date()}")
    day_slices.append((d.strftime("%Y-%m-%d"), day_df))

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

models = [
    ("ARX", "green"),
    ("Ridge", "blue"),
    ("DLinear", "orange"),
    ("LSTM", "red"),
]

fig = make_subplots(
    rows=4,
    cols=2,
    shared_xaxes="columns",
    shared_yaxes=True,
    subplot_titles=[day for day, _ in day_slices],
    vertical_spacing=0.06
)

for col, (day, day_df) in enumerate(day_slices, start=1):

    for row, (model, color) in enumerate(models, start=1):

        # --- Prediction interval ---
        fig.add_trace(
            go.Scatter(
                x=list(day_df["datetime"]) + list(day_df["datetime"][::-1]),
                y=list(day_df[f"{model}_upper"]) + list(day_df[f"{model}_lower"][::-1]),
                fill="toself",
                fillcolor=color,
                opacity=0.18,
                line=dict(width=0),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row, col=col
        )

        # --- Median forecast ---
        fig.add_trace(
            go.Scatter(
                x=day_df["datetime"],
                y=day_df[f"{model}_pred"],
                line=dict(color=color, width=2, dash="dash"),
                name=model,
                showlegend=(col == 1),
                legendgroup=model,
            ),
            row=row, col=col
        )

        # --- Actual ---
        fig.add_trace(
            go.Scatter(
                x=day_df["datetime"],
                y=day_df["Actual"],
                line=dict(color="black", width=2.5),
                name="Actual",
                showlegend=(row == 1 and col == 1),
                legendgroup="Actual",
            ),
            row=row, col=col
        )

        # --- Y-axis label per row ---
        if col == 1:
            fig.update_yaxes(
                title_text=f"{model}<br>Power (kW)",
                row=row,
                col=col
            )

    # --- X-axis formatting ---
    fig.update_xaxes(
        tickformat="%H:%M",
        dtick=2 * 60 * 60 * 1000,
        title_text="Time (h)",
        row=4,
        col=col
    )

fig.update_layout(
    height=1400,
    width=1800,
    template="simple_white",
    title=dict(
        text="Probabilistic Hour-Ahead PV Power Forecasts with ACI (Daily Reset)",
        x=0.5,
        font=dict(size=22)
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.08,
        xanchor="center",
        x=0.5,
        font=dict(size=18)
    ),
    font=dict(size=18)
)

for ann in fig.layout.annotations:
    ann.font.size = 18

fig.show()